Gym Management System

In [1]:
import tkinter as tk
from tkinter import ttk, messagebox
from datetime import date, datetime


CLASSES

In [2]:
class Member:
    def __init__(self, member_id, name, phone, trainer_id=None):
        self.member_id = member_id
        self.name = name
        self.phone = phone
        self.trainer_id = trainer_id 
     

In [3]:
class Trainer:
    def __init__(self, trainer_id, name, phone, specialty):
        self.trainer_id = trainer_id
        self.name = name
        self.phone = phone
        self.specialty = specialty


In [4]:
class Membership:
    def __init__(self, membership_id, member_id, membership_type,
                 price, start_date, end_date):
        self.membership_id = membership_id
        self.member_id = member_id
        self.membership_type = membership_type
        self.price = price
        self.start_date = start_date
        self.end_date = end_date

In [5]:
class Payment:
    def __init__(self, payment_id, member_id, amount, payment_date, method):
        self.payment_id = payment_id
        self.member_id = member_id
        self.amount = amount
        self.payment_date = payment_date
        self.method = method

In [6]:
class Attendance:
    def __init__(self, attendance_id, member_id, attendance_date, check_in):
        self.attendance_id = attendance_id
        self.member_id = member_id
        self.attendance_date = attendance_date
        self.check_in = check_in

In [7]:
class GymManagementSystem:
    def __init__(self):
        self.members = []
        self.trainers = []
        self.memberships = []
        self.payments = []
        self.attendance_records = []

        self.member_counter = 1
        self.trainer_counter = 1
        self.membership_counter = 1
        self.payment_counter = 1
        self.attendance_counter = 1

    def add_member(self, name, phone, trainer_id=None):
        if not name.strip():
            raise ValueError("Member name is required.")

        if trainer_id not in (None, ""):
            trainer_id = int(trainer_id)
            if not any(t.trainer_id == trainer_id for t in self.trainers):
                raise ValueError("Trainer ID does not exist.")

        member = Member(
            self.member_counter, name.strip(), phone.strip(),
            int(trainer_id) if trainer_id else None
        )
        self.members.append(member)
        self.member_counter += 1
        return member

    def update_member(self, member_id, name, phone, trainer_id=None):
        if not name.strip():
            raise ValueError("Member name is required.")

        if trainer_id not in (None, ""):
            trainer_id = int(trainer_id)
            if not any(t.trainer_id == trainer_id for t in self.trainers):
                raise ValueError("Trainer ID does not exist.")

        for member in self.members:
            if member.member_id == int(member_id):
                member.name = name.strip()
                member.phone = phone.strip()
                member.trainer_id = int(trainer_id) if trainer_id else None
                return True

        return False

    def delete_member(self, member_id):
        for member in self.members:
            if member.member_id == int(member_id):
                self.members.remove(member)
                return True
        return False

    def search_members(self, keyword):
        keyword = keyword.lower()
        return [
            member for member in self.members
            if keyword in member.name.lower()
            or keyword in member.phone.lower()
            or keyword == str(member.member_id)
        ]

    def add_trainer(self, name, phone, specialty):
        if not name.strip():
            raise ValueError("Trainer name is required.")

        trainer = Trainer(
            self.trainer_counter, name.strip(), phone.strip(),
            specialty.strip()
        )
        self.trainers.append(trainer)
        self.trainer_counter += 1
        return trainer

    def add_membership(self, member_id, membership_type, price,
                       start_date, duration_days):
        member_id = int(member_id)
        price = float(price)
        duration_days = int(duration_days)

        if not any(m.member_id == member_id for m in self.members):
            raise ValueError("Member ID does not exist.")
        if price < 0 or duration_days <= 0:
            raise ValueError("Price must be non-negative and duration positive.")

        start = date.fromisoformat(start_date)
        end = start + timedelta(days=duration_days)

        membership = Membership(
            self.membership_counter, member_id, membership_type,
            price, start.isoformat(), end.isoformat()
        )
        self.memberships.append(membership)
        self.membership_counter += 1
        return membership

    def add_payment(self, member_id, amount, method):
        member_id = int(member_id)
        amount = float(amount)

        if not any(m.member_id == member_id for m in self.members):
            raise ValueError("Member ID does not exist.")
        if amount <= 0:
            raise ValueError("Payment amount must be greater than zero.")

        payment = Payment(
            self.payment_counter, member_id, amount,
            date.today().isoformat(), method
        )
        self.payments.append(payment)
        self.payment_counter += 1
        return payment

    def record_attendance(self, member_id):
        member_id = int(member_id)

        if not any(m.member_id == member_id for m in self.members):
            raise ValueError("Member ID does not exist.")

        record = Attendance(
            self.attendance_counter, member_id,
            date.today().isoformat(),
            datetime.now().strftime("%H:%M:%S")
        )
        self.attendance_records.append(record)
        self.attendance_counter += 1
        return record

    def active_memberships(self):
        today = date.today().isoformat()
        return [
            m for m in self.memberships
            if m.start_date <= today <= m.end_date
        ]

    def dashboard(self):
        return {
            "Members": len(self.members),
            "Trainers": len(self.trainers),
            "Memberships": len(self.memberships),
            "Active memberships": len(self.active_memberships()),
            "Payments": len(self.payments),
            "Attendance records": len(self.attendance_records),
            "Total revenue": sum(p.amount for p in self.payments)
        }

Class for the GUI

In [8]:
class GymApp:
    BG = "#F3F6FB"
    NAVY = "#14213D"
    TEAL = "#12B8A6"
    WHITE = "#FFFFFF"
    TEXT = "#1F2937"
    MUTED = "#64748B"
    RED = "#DC2626"

    def __init__(self, root):
        self.root = root
        self.root.title("FitZone | Gym Management System")
        self.root.geometry("1250x780")
        self.root.minsize(1050, 650)
        self.root.configure(bg=self.BG)

        self.gym = GymManagementSystem()

        self.setup_styles()
        self.create_header()
        self.create_notebook()

        self.create_dashboard_tab()
        self.create_members_tab()
        self.create_trainers_tab()
        self.create_memberships_tab()
        self.create_payments_tab()
        self.create_attendance_tab()

        self.refresh_all()

    # ---------- STYLING ----------

    def setup_styles(self):
        style = ttk.Style()

        try:
            style.theme_use("clam")
        except tk.TclError:
            pass

        style.configure(
            "TNotebook",
            background=self.BG,
            borderwidth=0
        )

        style.configure(
            "TNotebook.Tab",
            background="#E2E8F0",
            foreground=self.NAVY,
            padding=(20, 12),
            font=("Segoe UI", 10, "bold")
        )

        style.map(
            "TNotebook.Tab",
            background=[("selected", self.TEAL)],
            foreground=[("selected", self.WHITE)]
        )

        style.configure(
            "TFrame",
            background=self.BG
        )

        style.configure(
            "Card.TFrame",
            background=self.WHITE
        )

        style.configure(
            "TLabel",
            background=self.BG,
            foreground=self.TEXT,
            font=("Segoe UI", 10)
        )

        style.configure(
            "Header.TLabel",
            background=self.NAVY,
            foreground=self.WHITE,
            font=("Segoe UI", 23, "bold")
        )

        style.configure(
            "Subtitle.TLabel",
            background=self.NAVY,
            foreground="#CBD5E1",
            font=("Segoe UI", 10)
        )

        style.configure(
            "CardTitle.TLabel",
            background=self.WHITE,
            foreground=self.MUTED,
            font=("Segoe UI", 10, "bold")
        )

        style.configure(
            "CardValue.TLabel",
            background=self.WHITE,
            foreground=self.NAVY,
            font=("Segoe UI", 23, "bold")
        )

        style.configure(
            "Section.TLabel",
            background=self.BG,
            foreground=self.NAVY,
            font=("Segoe UI", 17, "bold")
        )

        style.configure(
            "TButton",
            font=("Segoe UI", 10, "bold"),
            padding=(12, 8),
            background=self.TEAL,
            foreground=self.WHITE,
            borderwidth=0
        )

        style.map(
            "TButton",
            background=[("active", "#0F988A"), ("disabled", "#CBD5E1")],
            foreground=[("disabled", "#64748B")]
        )

        style.configure(
            "Danger.TButton",
            background=self.RED,
            foreground=self.WHITE
        )

        style.map(
            "Danger.TButton",
            background=[("active", "#B91C1C")]
        )

        style.configure(
            "Secondary.TButton",
            background="#E2E8F0",
            foreground=self.NAVY
        )

        style.map(
            "Secondary.TButton",
            background=[("active", "#CBD5E1")]
        )

        style.configure(
            "Treeview",
            background=self.WHITE,
            fieldbackground=self.WHITE,
            foreground=self.TEXT,
            rowheight=32,
            font=("Segoe UI", 10),
            borderwidth=0
        )

        style.configure(
            "Treeview.Heading",
            background=self.NAVY,
            foreground=self.WHITE,
            font=("Segoe UI", 10, "bold"),
            padding=10,
            relief="flat"
        )

        style.map(
            "Treeview",
            background=[("selected", "#CCFBF1")],
            foreground=[("selected", self.NAVY)]
        )

        style.configure(
            "TLabelframe",
            background=self.BG,
            bordercolor="#D8E0EB"
        )

        style.configure(
            "TLabelframe.Label",
            background=self.BG,
            foreground=self.NAVY,
            font=("Segoe UI", 11, "bold")
        )

        style.configure(
            "TEntry",
            padding=7
        )

        style.configure(
            "TCombobox",
            padding=6
        )

    def create_header(self):
        header = tk.Frame(self.root, bg=self.NAVY, height=100)
        header.pack(fill="x")
        header.pack_propagate(False)

        left = tk.Frame(header, bg=self.NAVY)
        left.pack(side="left", padx=28, pady=15)

        tk.Label(
            left,
            text="FITZONE",
            font=("Segoe UI", 25, "bold"),
            bg=self.NAVY,
            fg=self.TEAL
        ).pack(anchor="w")

        tk.Label(
            left,
            text="GYM MANAGEMENT SYSTEM",
            font=("Segoe UI", 10),
            bg=self.NAVY,
            fg="#CBD5E1"
        ).pack(anchor="w")

        right = tk.Frame(header, bg=self.NAVY)
        right.pack(side="right", padx=28)

        tk.Label(
            right,
            text="MANAGEMENT PORTAL",
            font=("Segoe UI", 10, "bold"),
            bg=self.NAVY,
            fg=self.WHITE
        ).pack(anchor="e", pady=(25, 3))

        tk.Label(
            right,
            text=date.today().strftime("%d %B %Y"),
            font=("Segoe UI", 10),
            bg=self.NAVY,
            fg="#CBD5E1"
        ).pack(anchor="e")

    def create_notebook(self):
        self.notebook = ttk.Notebook(self.root)
        self.notebook.pack(fill="both", expand=True, padx=20, pady=18)

        self.dashboard_tab = ttk.Frame(self.notebook, padding=18)
        self.members_tab = ttk.Frame(self.notebook, padding=18)
        self.trainers_tab = ttk.Frame(self.notebook, padding=18)
        self.memberships_tab = ttk.Frame(self.notebook, padding=18)
        self.payments_tab = ttk.Frame(self.notebook, padding=18)
        self.attendance_tab = ttk.Frame(self.notebook, padding=18)

        self.notebook.add(self.dashboard_tab, text="  Dashboard  ")
        self.notebook.add(self.members_tab, text="  Members  ")
        self.notebook.add(self.trainers_tab, text="  Trainers  ")
        self.notebook.add(self.memberships_tab, text="  Memberships  ")
        self.notebook.add(self.payments_tab, text="  Payments  ")
        self.notebook.add(self.attendance_tab, text="  Attendance  ")

        footer = tk.Frame(self.root, bg="#E2E8F0", height=30)
        footer.pack(fill="x", side="bottom")

        tk.Label(
            footer,
            text="FITZONE  |  GYM MANAGEMENT",
            bg="#E2E8F0",
            fg=self.MUTED,
            font=("Segoe UI", 9)
        ).pack(side="left", padx=20, pady=6)

        tk.Label(
            footer,
            text="SYSTEM ONLINE",
            bg="#E2E8F0",
            fg="#0F988A",
            font=("Segoe UI", 9, "bold")
        ).pack(side="right", padx=20, pady=6)

    # ---------- GENERAL HELPERS ----------

    def make_tree(self, parent, columns, headings, widths=None):
        container = ttk.Frame(parent)
        container.pack(fill="both", expand=True, pady=(10, 0))

        tree = ttk.Treeview(
            container,
            columns=columns,
            show="headings",
            selectmode="browse"
        )

        if widths is None:
            widths = [130] * len(columns)

        for col, heading, width in zip(columns, headings, widths):
            tree.heading(col, text=heading)
            tree.column(col, width=width, minwidth=70, anchor="center")

        y_scroll = ttk.Scrollbar(
            container,
            orient="vertical",
            command=tree.yview
        )

        x_scroll = ttk.Scrollbar(
            container,
            orient="horizontal",
            command=tree.xview
        )

        tree.configure(
            yscrollcommand=y_scroll.set,
            xscrollcommand=x_scroll.set
        )

        tree.grid(row=0, column=0, sticky="nsew")
        y_scroll.grid(row=0, column=1, sticky="ns")
        x_scroll.grid(row=1, column=0, sticky="ew")

        container.rowconfigure(0, weight=1)
        container.columnconfigure(0, weight=1)

        tree.tag_configure("even", background="#FFFFFF")
        tree.tag_configure("odd", background="#F1F5F9")

        return tree

    def fill_tree(self, tree, rows):
        tree.delete(*tree.get_children())

        for index, row in enumerate(rows):
            tag = "even" if index % 2 == 0 else "odd"
            tree.insert("", "end", values=row, tags=(tag,))

    def get_selected_id(self, tree):
        selected = tree.selection()

        if not selected:
            messagebox.showwarning(
                "No Selection",
                "Please select a record from the table."
            )
            return None

        return tree.item(selected[0], "values")[0]

    def member_name(self, member_id):
        for member in self.gym.members:
            if member.member_id == int(member_id):
                return member.name
        return "Unknown member"

    def trainer_name(self, trainer_id):
        if trainer_id in (None, ""):
            return "Not assigned"

        for trainer in self.gym.trainers:
            if trainer.trainer_id == int(trainer_id):
                return trainer.name

        return "Not assigned"

    def member_options(self):
        return [
            f"{m.member_id} - {m.name}"
            for m in self.gym.members
        ]

    def trainer_options(self):
        return [
            f"{t.trainer_id} - {t.name}"
            for t in self.gym.trainers
        ]

    def parse_option_id(self, value):
        if not value.strip():
            return None

        return int(value.split(" - ", 1)[0])

    def refresh_all(self):
        self.refresh_dashboard()
        self.refresh_members()
        self.refresh_trainers()
        self.refresh_memberships()
        self.refresh_payments()
        self.refresh_attendance()
        self.refresh_dropdowns()

    def refresh_dropdowns(self):
        member_values = self.member_options()
        trainer_values = self.trainer_options()

        if hasattr(self, "member_trainer_combo"):
            self.member_trainer_combo["values"] = [""] + trainer_values

        if hasattr(self, "membership_member_combo"):
            self.membership_member_combo["values"] = member_values

        if hasattr(self, "payment_member_combo"):
            self.payment_member_combo["values"] = member_values

        if hasattr(self, "attendance_member_combo"):
            self.attendance_member_combo["values"] = member_values

    # ---------- DASHBOARD ----------

    def create_dashboard_tab(self):
        ttk.Label(
            self.dashboard_tab,
            text="Dashboard Overview",
            style="Section.TLabel"
        ).pack(anchor="w")

        ttk.Label(
            self.dashboard_tab,
            text="A quick overview of your gym's activity and performance."
        ).pack(anchor="w", pady=(3, 18))

        cards_frame = ttk.Frame(self.dashboard_tab)
        cards_frame.pack(fill="x")

        card_definitions = [
            ("Members", "Total registered members"),
            ("Trainers", "Available trainers"),
            ("Active memberships", "Currently active"),
            ("Total revenue", "Recorded payments")
        ]

        self.dashboard_values = {}

        for index, (key, description) in enumerate(card_definitions):
            card = tk.Frame(
                cards_frame,
                bg=self.WHITE,
                highlightbackground="#DCE4EF",
                highlightthickness=1
            )

            card.grid(
                row=0,
                column=index,
                sticky="nsew",
                padx=6,
                pady=6
            )

            cards_frame.columnconfigure(index, weight=1)

            tk.Frame(
                card,
                bg=self.TEAL,
                height=5
            ).pack(fill="x")

            inner = tk.Frame(card, bg=self.WHITE)
            inner.pack(fill="both", expand=True, padx=15, pady=15)

            tk.Label(
                inner,
                text=key.upper(),
                bg=self.WHITE,
                fg=self.MUTED,
                font=("Segoe UI", 9, "bold")
            ).pack(anchor="w")

            value_label = tk.Label(
                inner,
                text="0",
                bg=self.WHITE,
                fg=self.NAVY,
                font=("Segoe UI", 25, "bold")
            )
            value_label.pack(anchor="w", pady=(8, 2))

            tk.Label(
                inner,
                text=description,
                bg=self.WHITE,
                fg=self.MUTED,
                font=("Segoe UI", 9)
            ).pack(anchor="w")

            self.dashboard_values[key] = value_label

        lower = ttk.Frame(self.dashboard_tab)
        lower.pack(fill="both", expand=True, pady=(22, 0))

        summary = ttk.LabelFrame(
            lower,
            text="Gym Summary",
            padding=18
        )
        summary.pack(side="left", fill="both", expand=True, padx=(0, 10))

        self.summary_text = tk.Text(
            summary,
            height=10,
            bg=self.WHITE,
            fg=self.TEXT,
            font=("Segoe UI", 11),
            relief="flat",
            padx=12,
            pady=12,
            state="disabled"
        )
        self.summary_text.pack(fill="both", expand=True)

        actions = ttk.LabelFrame(
            lower,
            text="Quick Actions",
            padding=18
        )
        actions.pack(side="left", fill="y", padx=(10, 0))

        ttk.Button(
            actions,
            text="+ Add Member",
            command=lambda: self.notebook.select(self.members_tab)
        ).pack(fill="x", pady=7)

        ttk.Button(
            actions,
            text="+ Add Trainer",
            command=lambda: self.notebook.select(self.trainers_tab)
        ).pack(fill="x", pady=7)

        ttk.Button(
            actions,
            text="+ New Membership",
            command=lambda: self.notebook.select(self.memberships_tab)
        ).pack(fill="x", pady=7)

        ttk.Button(
            actions,
            text="Record Attendance",
            command=lambda: self.notebook.select(self.attendance_tab)
        ).pack(fill="x", pady=7)

        ttk.Button(
            actions,
            text="Refresh Dashboard",
            style="Secondary.TButton",
            command=self.refresh_all
        ).pack(fill="x", pady=(20, 7))

    def refresh_dashboard(self):
        stats = self.gym.dashboard()

        if hasattr(self, "dashboard_values"):
            for key, label in self.dashboard_values.items():
                value = stats.get(key, 0)

                if key == "Total revenue":
                    value = f"{value:,.2f}"

                label.config(text=str(value))

        if hasattr(self, "summary_text"):
            active_count = len(self.gym.active_memberships())
            expired_count = sum(
                1 for m in self.gym.memberships
                if m.end_date < date.today().isoformat()
            )

            text = (
                f"Total members:          {len(self.gym.members)}\n\n"
                f"Total trainers:          {len(self.gym.trainers)}\n\n"
                f"Total memberships:       {len(self.gym.memberships)}\n\n"
                f"Active memberships:      {active_count}\n\n"
                f"Expired memberships:     {expired_count}\n\n"
                f"Recorded payments:       {len(self.gym.payments)}\n\n"
                f"Attendance records:      {len(self.gym.attendance_records)}"
            )

            self.summary_text.config(state="normal")
            self.summary_text.delete("1.0", "end")
            self.summary_text.insert("1.0", text)
            self.summary_text.config(state="disabled")

    # ---------- MEMBERS ----------

    def create_members_tab(self):
        ttk.Label(
            self.members_tab,
            text="Member Management",
            style="Section.TLabel"
        ).pack(anchor="w")

        form = ttk.LabelFrame(
            self.members_tab,
            text="Member Information",
            padding=12
        )
        form.pack(fill="x", pady=(12, 10))

        self.member_name_var = tk.StringVar()
        self.member_phone_var = tk.StringVar()
        self.member_trainer_var = tk.StringVar()
        self.member_search_var = tk.StringVar()

        ttk.Label(form, text="Full Name").grid(
            row=0, column=0, sticky="w", padx=6, pady=5
        )
        ttk.Entry(
            form,
            textvariable=self.member_name_var,
            width=24
        ).grid(row=1, column=0, padx=6, pady=5, sticky="ew")

        ttk.Label(form, text="Phone Number").grid(
            row=0, column=1, sticky="w", padx=6, pady=5
        )
        ttk.Entry(
            form,
            textvariable=self.member_phone_var,
            width=20
        ).grid(row=1, column=1, padx=6, pady=5, sticky="ew")

        ttk.Label(form, text="Assigned Trainer").grid(
            row=0, column=2, sticky="w", padx=6, pady=5
        )
        self.member_trainer_combo = ttk.Combobox(
            form,
            textvariable=self.member_trainer_var,
            state="readonly",
            width=22
        )
        self.member_trainer_combo.grid(
            row=1, column=2, padx=6, pady=5, sticky="ew"
        )

        buttons = ttk.Frame(form)
        buttons.grid(row=1, column=3, padx=8, pady=5)

        ttk.Button(
            buttons, text="Add Member", command=self.add_member
        ).pack(side="left", padx=3)

        ttk.Button(
            buttons, text="Update", command=self.update_member
        ).pack(side="left", padx=3)

        ttk.Button(
            buttons,
            text="Delete",
            style="Danger.TButton",
            command=self.delete_member
        ).pack(side="left", padx=3)

        for col in range(3):
            form.columnconfigure(col, weight=1)

        search_frame = ttk.Frame(self.members_tab)
        search_frame.pack(fill="x", pady=(5, 0))

        ttk.Label(search_frame, text="Search members:").pack(side="left")

        ttk.Entry(
            search_frame,
            textvariable=self.member_search_var,
            width=30
        ).pack(side="left", padx=8)

        ttk.Button(
            search_frame,
            text="Search",
            command=self.search_members
        ).pack(side="left", padx=3)

        ttk.Button(
            search_frame,
            text="Show All",
            style="Secondary.TButton",
            command=self.refresh_members
        ).pack(side="left", padx=3)

        self.members_tree = self.make_tree(
            self.members_tab,
            ("id", "name", "phone", "trainer"),
            ("Member ID", "Full Name", "Phone", "Assigned Trainer"),
            (100, 260, 180, 240)
        )

        self.members_tree.bind(
            "<<TreeviewSelect>>",
            self.select_member
        )

    def add_member(self):
        try:
            trainer_id = self.parse_option_id(
                self.member_trainer_var.get()
            )

            member = self.gym.add_member(
                self.member_name_var.get(),
                self.member_phone_var.get(),
                trainer_id
            )

            self.member_name_var.set("")
            self.member_phone_var.set("")
            self.member_trainer_var.set("")

            self.refresh_all()
            messagebox.showinfo(
                "Success",
                f"Member {member.name} added successfully."
            )

        except (ValueError, TypeError) as error:
            messagebox.showerror("Invalid Information", str(error))

    def update_member(self):
        member_id = self.get_selected_id(self.members_tree)

        if member_id is None:
            return

        try:
            trainer_id = self.parse_option_id(
                self.member_trainer_var.get()
            )

            updated = self.gym.update_member(
                member_id,
                self.member_name_var.get(),
                self.member_phone_var.get(),
                trainer_id
            )

            if updated:
                self.refresh_all()
                messagebox.showinfo(
                    "Success",
                    "Member information updated."
                )
            else:
                messagebox.showerror("Error", "Member not found.")

        except (ValueError, TypeError) as error:
            messagebox.showerror("Invalid Information", str(error))

    def delete_member(self):
        member_id = self.get_selected_id(self.members_tree)

        if member_id is None:
            return

        if not messagebox.askyesno(
            "Confirm Delete",
            "Delete this member? Related historical records will remain."
        ):
            return

        if self.gym.delete_member(member_id):
            self.refresh_all()
            self.member_name_var.set("")
            self.member_phone_var.set("")
            self.member_trainer_var.set("")
            messagebox.showinfo("Success", "Member deleted.")
        else:
            messagebox.showerror("Error", "Member not found.")

    def select_member(self, event=None):
        selected = self.members_tree.selection()

        if not selected:
            return

        values = self.members_tree.item(selected[0], "values")
        member_id = int(values[0])

        member = next(
            (m for m in self.gym.members if m.member_id == member_id),
            None
        )

        if member:
            self.member_name_var.set(member.name)
            self.member_phone_var.set(member.phone)
            self.member_trainer_var.set(
                "" if member.trainer_id is None
                else next(
                    (
                        f"{t.trainer_id} - {t.name}"
                        for t in self.gym.trainers
                        if t.trainer_id == member.trainer_id
                    ),
                    ""
                )
            )

    def search_members(self):
        keyword = self.member_search_var.get().strip()
        results = self.gym.search_members(keyword)

        rows = [
            (
                m.member_id,
                m.name,
                m.phone,
                self.trainer_name(m.trainer_id)
            )
            for m in results
        ]

        self.fill_tree(self.members_tree, rows)

    def refresh_members(self):
        if not hasattr(self, "members_tree"):
            return

        rows = [
            (
                m.member_id,
                m.name,
                m.phone,
                self.trainer_name(m.trainer_id)
            )
            for m in self.gym.members
        ]

        self.fill_tree(self.members_tree, rows)

    # ---------- TRAINERS ----------

    def create_trainers_tab(self):
        ttk.Label(
            self.trainers_tab,
            text="Trainer Management",
            style="Section.TLabel"
        ).pack(anchor="w")

        form = ttk.LabelFrame(
            self.trainers_tab,
            text="Trainer Information",
            padding=12
        )
        form.pack(fill="x", pady=(12, 10))

        self.trainer_name_var = tk.StringVar()
        self.trainer_phone_var = tk.StringVar()
        self.trainer_specialty_var = tk.StringVar()

        fields = [
            ("Full Name", self.trainer_name_var),
            ("Phone Number", self.trainer_phone_var),
            ("Specialty", self.trainer_specialty_var)
        ]

        for col, (label, variable) in enumerate(fields):
            ttk.Label(form, text=label).grid(
                row=0, column=col, sticky="w", padx=6, pady=5
            )
            ttk.Entry(
                form,
                textvariable=variable,
                width=25
            ).grid(row=1, column=col, padx=6, pady=5, sticky="ew")
            form.columnconfigure(col, weight=1)

        ttk.Button(
            form,
            text="Add Trainer",
            command=self.add_trainer
        ).grid(row=1, column=3, padx=10, pady=5)

        self.trainers_tree = self.make_tree(
            self.trainers_tab,
            ("id", "name", "phone", "specialty"),
            ("Trainer ID", "Full Name", "Phone", "Specialty"),
            (100, 260, 200, 250)
        )

    def add_trainer(self):
        try:
            trainer = self.gym.add_trainer(
                self.trainer_name_var.get(),
                self.trainer_phone_var.get(),
                self.trainer_specialty_var.get()
            )

            self.trainer_name_var.set("")
            self.trainer_phone_var.set("")
            self.trainer_specialty_var.set("")

            self.refresh_all()

            messagebox.showinfo(
                "Success",
                f"Trainer {trainer.name} added successfully."
            )

        except (ValueError, TypeError) as error:
            messagebox.showerror("Invalid Information", str(error))

    def refresh_trainers(self):
        if not hasattr(self, "trainers_tree"):
            return

        rows = [
            (t.trainer_id, t.name, t.phone, t.specialty)
            for t in self.gym.trainers
        ]

        self.fill_tree(self.trainers_tree, rows)

    # ---------- MEMBERSHIPS ----------

    def create_memberships_tab(self):
        ttk.Label(
            self.memberships_tab,
            text="Membership Management",
            style="Section.TLabel"
        ).pack(anchor="w")

        form = ttk.LabelFrame(
            self.memberships_tab,
            text="Create Membership",
            padding=12
        )
        form.pack(fill="x", pady=(12, 10))

        self.membership_member_var = tk.StringVar()
        self.membership_type_var = tk.StringVar(value="Monthly")
        self.membership_price_var = tk.StringVar()
        self.membership_duration_var = tk.StringVar(value="30")

        ttk.Label(form, text="Member").grid(
            row=0, column=0, sticky="w", padx=6, pady=5
        )
        self.membership_member_combo = ttk.Combobox(
            form,
            textvariable=self.membership_member_var,
            state="readonly",
            width=22
        )
        self.membership_member_combo.grid(
            row=1, column=0, padx=6, pady=5, sticky="ew"
        )

        ttk.Label(form, text="Plan").grid(
            row=0, column=1, sticky="w", padx=6, pady=5
        )
        ttk.Combobox(
            form,
            textvariable=self.membership_type_var,
            values=("Monthly", "Quarterly", "Yearly", "Weekly", "Custom"),
            state="readonly",
            width=17
        ).grid(row=1, column=1, padx=6, pady=5, sticky="ew")

        ttk.Label(form, text="Price").grid(
            row=0, column=2, sticky="w", padx=6, pady=5
        )
        ttk.Entry(
            form,
            textvariable=self.membership_price_var,
            width=15
        ).grid(row=1, column=2, padx=6, pady=5, sticky="ew")

        ttk.Label(form, text="Duration (days)").grid(
            row=0, column=3, sticky="w", padx=6, pady=5
        )
        ttk.Entry(
            form,
            textvariable=self.membership_duration_var,
            width=15
        ).grid(row=1, column=3, padx=6, pady=5, sticky="ew")

        ttk.Button(
            form,
            text="Create Membership",
            command=self.add_membership
        ).grid(row=1, column=4, padx=10, pady=5)

        for col in range(4):
            form.columnconfigure(col, weight=1)

        self.memberships_tree = self.make_tree(
            self.memberships_tab,
            ("id", "member_id", "member", "type", "price",
             "start", "end", "status"),
            ("ID", "Member ID", "Member", "Plan", "Price",
             "Start Date", "End Date", "Status"),
            (65, 85, 180, 110, 100, 120, 120, 110)
        )

    def add_membership(self):
        try:
            member_id = self.parse_option_id(
                self.membership_member_var.get()
            )

            if member_id is None:
                raise ValueError("Please select a member.")

            membership = self.gym.add_membership(
                member_id,
                self.membership_type_var.get(),
                self.membership_price_var.get(),
                date.today().isoformat(),
                self.membership_duration_var.get()
            )

            self.membership_price_var.set("")

            self.refresh_all()

            messagebox.showinfo(
                "Success",
                f"Membership created. Expires on {membership.end_date}."
            )

        except (ValueError, TypeError) as error:
            messagebox.showerror("Invalid Information", str(error))

    def refresh_memberships(self):
        if not hasattr(self, "memberships_tree"):
            return

        today = date.today().isoformat()
        rows = []

        for m in self.gym.memberships:
            if m.start_date <= today <= m.end_date:
                status = "Active"
            elif today < m.start_date:
                status = "Upcoming"
            else:
                status = "Expired"

            rows.append((
                m.membership_id,
                m.member_id,
                self.member_name(m.member_id),
                m.membership_type,
                f"{m.price:,.2f}",
                m.start_date,
                m.end_date,
                status
            ))

        self.fill_tree(self.memberships_tree, rows)

    # ---------- PAYMENTS ----------

    def create_payments_tab(self):
        ttk.Label(
            self.payments_tab,
            text="Payment Management",
            style="Section.TLabel"
        ).pack(anchor="w")

        form = ttk.LabelFrame(
            self.payments_tab,
            text="Record Payment",
            padding=12
        )
        form.pack(fill="x", pady=(12, 10))

        self.payment_member_var = tk.StringVar()
        self.payment_amount_var = tk.StringVar()
        self.payment_method_var = tk.StringVar(value="Cash")

        ttk.Label(form, text="Member").grid(
            row=0, column=0, sticky="w", padx=6, pady=5
        )
        self.payment_member_combo = ttk.Combobox(
            form,
            textvariable=self.payment_member_var,
            state="readonly",
            width=24
        )
        self.payment_member_combo.grid(
            row=1, column=0, padx=6, pady=5, sticky="ew"
        )

        ttk.Label(form, text="Amount").grid(
            row=0, column=1, sticky="w", padx=6, pady=5
        )
        ttk.Entry(
            form,
            textvariable=self.payment_amount_var,
            width=20
        ).grid(row=1, column=1, padx=6, pady=5, sticky="ew")

        ttk.Label(form, text="Payment Method").grid(
            row=0, column=2, sticky="w", padx=6, pady=5
        )
        ttk.Combobox(
            form,
            textvariable=self.payment_method_var,
            values=("Cash", "Credit Card", "Debit Card", "Bank Transfer", "Other"),
            state="readonly",
            width=20
        ).grid(row=1, column=2, padx=6, pady=5, sticky="ew")

        ttk.Button(
            form,
            text="Record Payment",
            command=self.add_payment
        ).grid(row=1, column=3, padx=10, pady=5)

        for col in range(3):
            form.columnconfigure(col, weight=1)

        self.payments_tree = self.make_tree(
            self.payments_tab,
            ("id", "member_id", "member", "amount", "date", "method"),
            ("Payment ID", "Member ID", "Member", "Amount",
             "Payment Date", "Method"),
            (100, 100, 220, 130, 160, 160)
        )

    def add_payment(self):
        try:
            member_id = self.parse_option_id(
                self.payment_member_var.get()
            )

            if member_id is None:
                raise ValueError("Please select a member.")

            payment = self.gym.add_payment(
                member_id,
                self.payment_amount_var.get(),
                self.payment_method_var.get()
            )

            self.payment_amount_var.set("")
            self.refresh_all()

            messagebox.showinfo(
                "Success",
                f"Payment of {payment.amount:,.2f} recorded."
            )

        except (ValueError, TypeError) as error:
            messagebox.showerror("Invalid Information", str(error))

    def refresh_payments(self):
        if not hasattr(self, "payments_tree"):
            return

        rows = [
            (
                p.payment_id,
                p.member_id,
                self.member_name(p.member_id),
                f"{p.amount:,.2f}",
                p.payment_date,
                p.method
            )
            for p in self.gym.payments
        ]

        self.fill_tree(self.payments_tree, rows)

    # ---------- ATTENDANCE ----------

    def create_attendance_tab(self):
        ttk.Label(
            self.attendance_tab,
            text="Attendance Management",
            style="Section.TLabel"
        ).pack(anchor="w")

        form = ttk.LabelFrame(
            self.attendance_tab,
            text="Check In Member",
            padding=12
        )
        form.pack(fill="x", pady=(12, 10))

        self.attendance_member_var = tk.StringVar()

        ttk.Label(form, text="Select Member").pack(
            side="left", padx=8
        )

        self.attendance_member_combo = ttk.Combobox(
            form,
            textvariable=self.attendance_member_var,
            state="readonly",
            width=32
        )
        self.attendance_member_combo.pack(side="left", padx=8)

        ttk.Button(
            form,
            text="Record Check-In",
            command=self.add_attendance
        ).pack(side="left", padx=8)

        ttk.Button(
            form,
            text="Refresh",
            style="Secondary.TButton",
            command=self.refresh_attendance
        ).pack(side="left", padx=8)

        self.attendance_tree = self.make_tree(
            self.attendance_tab,
            ("id", "member_id", "member", "date", "time"),
            ("Attendance ID", "Member ID", "Member",
             "Attendance Date", "Check-In Time"),
            (120, 110, 250, 180, 180)
        )

    def add_attendance(self):
        try:
            member_id = self.parse_option_id(
                self.attendance_member_var.get()
            )

            if member_id is None:
                raise ValueError("Please select a member.")

            record = self.gym.record_attendance(member_id)
            self.refresh_all()

            messagebox.showinfo(
                "Check-In Successful",
                f"{self.member_name(record.member_id)} checked in at "
                f"{record.check_in}."
            )

        except (ValueError, TypeError) as error:
            messagebox.showerror("Invalid Information", str(error))

    def refresh_attendance(self):
        if not hasattr(self, "attendance_tree"):
            return

        rows = [
            (
                a.attendance_id,
                a.member_id,
                self.member_name(a.member_id),
                a.attendance_date,
                a.check_in
            )
            for a in self.gym.attendance_records
        ]

        self.fill_tree(self.attendance_tree, rows)


# ==================== RUN APPLICATION ====================

if __name__ == "__main__":
    root = tk.Tk()
    app = GymApp(root)
    root.mainloop()


Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Users\Abdelrahman\AppData\Local\Programs\Python\Python312\Lib\tkinter\__init__.py", line 1948, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\Abdelrahman\AppData\Local\Temp\ipykernel_25876\2381756493.py", line 1000, in add_membership
    membership = self.gym.add_membership(
                 ^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Abdelrahman\AppData\Local\Temp\ipykernel_25876\1604687977.py", line 90, in add_membership
    end = start + timedelta(days=duration_days)
                  ^^^^^^^^^
NameError: name 'timedelta' is not defined
Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Users\Abdelrahman\AppData\Local\Programs\Python\Python312\Lib\tkinter\__init__.py", line 1948, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\Abdelrahman\AppData\Local\Temp\ipykernel_25876\2381756493.py", line 1000, in add_membe